# 3.3 SES Prototype — Beam Handover (Reproducible GitHub Version)
**Purpose:** Prepare handover anomaly artefacts for the Streamlit dashboard using the existing GitHub repo structure.

**Primary input (repo):** `data/processed/handover_table.csv` (if present)

**Outputs (repo):**
- `data/processed/handover_table.csv` (ensured)
- `reports/figures/handover_event_shap_values.csv`
- `reports/figures/handover_event_heatmap.png`
- `reports/figures/handover_continuous_heatmap.png`

**Implementation note:** The dashboard expects a SHAP *matrix* (rows=features, cols=event steps) even though handover data is event-based. This notebook therefore treats a sequence of handover events as an ordered window and explains a **handover risk score** via a surrogate model.

In [ ]:
# ============================================================
# 0) Imports
# ============================================================
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor

import matplotlib.pyplot as plt
import shap


In [ ]:
# ============================================================
# 1) Resolve repo paths (DO NOT change repo tree)
# ============================================================
HERE = Path.cwd().resolve()
REPO = HERE
while not (REPO / "app.py").exists() and REPO != REPO.parent:
    REPO = REPO.parent

assert (REPO / "app.py").exists(), f"Repo root not found. Current working dir: {HERE}"

DATA_DIR = REPO / "data" / "processed"
FIG_DIR  = REPO / "reports" / "figures"
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

IN_HO = DATA_DIR / "handover_table.csv"

print("REPO    :", REPO)
print("DATA_DIR:", DATA_DIR)
print("FIG_DIR :", FIG_DIR)
print("IN_HO   :", IN_HO)


In [ ]:
# ============================================================
# 2) Load or create handover_table.csv
# ============================================================
def synth_handover_table(n: int = 1200) -> pd.DataFrame:
    rng = np.random.default_rng(42)
    t = pd.date_range("2021-11-01", periods=n, freq="5min", tz="UTC")
    beam_ids = rng.choice(["BEAM-01","BEAM-02","BEAM-03","BEAM-04"], size=n, replace=True)

    # Synthetic throughput before/after and recovery time
    thr_before = rng.uniform(10, 120, size=n)
    # occasional drops
    drop = rng.normal(0, 0.02, size=n)
    spikes = rng.choice(np.arange(n), size=max(10, n//60), replace=False)
    drop[spikes] += rng.uniform(-0.25, -0.08, size=len(spikes))  # sharp negative drops
    thr_after = thr_before * (1.0 + drop).clip(0.1, 1.2)

    recovery_s = rng.gamma(shape=2.0, scale=40.0, size=n)  # mostly < few minutes
    recovery_s[spikes] *= rng.uniform(3, 8, size=len(spikes))  # slow recovery for anomalies

    df = pd.DataFrame({
        "t": t,
        "beam_id": beam_ids,
        "throughput_before": thr_before,
        "throughput_after": thr_after,
        "recovery_time_s": recovery_s,
    })
    df["drop_pct"] = (df["throughput_after"] - df["throughput_before"]) / (df["throughput_before"] + 1e-9)
    return df

if IN_HO.exists():
    ho = pd.read_csv(IN_HO)
    # Normalise expected columns
    if "t" not in ho.columns:
        for cand in ("time", "timestamp", "datetime"):
            if cand in ho.columns:
                ho = ho.rename(columns={cand: "t"})
                break
    ho["t"] = pd.to_datetime(ho["t"], errors="coerce", utc=True)

    if "beam_id" not in ho.columns:
        for cand in ("beam", "beamId", "beam_id_series"):
            if cand in ho.columns:
                ho = ho.rename(columns={cand: "beam_id"})
                break
    if "beam_id" not in ho.columns:
        ho["beam_id"] = "BEAM"

    # Ensure drop_pct exists (dashboard uses it)
    if "drop_pct" not in ho.columns:
        if {"throughput_before", "throughput_after"}.issubset(set(ho.columns)):
            ho["drop_pct"] = (ho["throughput_after"] - ho["throughput_before"]) / (ho["throughput_before"] + 1e-9)
        else:
            # fallback proxy
            ho["drop_pct"] = 0.0

    # Add optional features used for SHAP if missing
    if "throughput_before" not in ho.columns:
        ho["throughput_before"] = 1.0
    if "throughput_after" not in ho.columns:
        ho["throughput_after"] = ho["throughput_before"] * (1.0 + ho["drop_pct"])
    if "recovery_time_s" not in ho.columns:
        ho["recovery_time_s"] = 60.0

    ho = ho.replace([np.inf, -np.inf], np.nan).fillna(method="ffill").fillna(0.0)
    ho = ho.sort_values("t").reset_index(drop=True)
else:
    ho = synth_handover_table()

# Persist ensured table for dashboard
ho.to_csv(IN_HO, index=False)
print("Saved/ensured:", IN_HO, "rows:", len(ho))
ho.head()


In [ ]:
# ============================================================
# 3) Define a handover risk score (event-level) for explanation
# ============================================================
# Conservative, interpretable proxy:
# - larger negative drop_pct and larger recovery_time_s => higher risk
drop_abs = np.clip(-ho["drop_pct"].values, 0, None)  # only penalise drops
recovery = ho["recovery_time_s"].values

# Normalise components
drop_n = (drop_abs - drop_abs.min()) / (drop_abs.max() - drop_abs.min() + 1e-12)
rec_n  = (recovery - recovery.min()) / (recovery.max() - recovery.min() + 1e-12)

risk_score = 0.7 * drop_n + 0.3 * rec_n
ho["risk_score"] = risk_score

print("Risk score summary:")
print(ho["risk_score"].describe())


## SHAP artefacts for the dashboard
The Streamlit app loads:
- `reports/figures/handover_event_shap_values.csv`
- `reports/figures/handover_event_heatmap.png`
- `reports/figures/handover_continuous_heatmap.png`

Here, SHAP explains a **surrogate model** trained to approximate `risk_score`.

In [ ]:
# ============================================================
# 4) Train surrogate model + compute SHAP values
# ============================================================
feature_cols = ["throughput_before", "throughput_after", "drop_pct", "recovery_time_s"]
X = ho[feature_cols].values
y = ho["risk_score"].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

rf = RandomForestRegressor(
    n_estimators=500,
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_scaled, y)

explainer = shap.TreeExplainer(rf)

# Select an "event window" around the highest-risk event
peak_idx = int(np.argmax(y))
win = 30  # events window
start = max(0, peak_idx - win//2)
end = min(len(ho), start + win)
idxs = np.arange(start, end)

X_win = X_scaled[idxs]                 # (events, features)
shap_vals = np.asarray(explainer.shap_values(X_win))  # (events, features)

# Convert to (features, events) to match dashboard loader
shap_matrix = shap_vals.T
time_labels = [f"e{i}" for i in range(shap_matrix.shape[1])]

def save_shap_matrix_csv(out_csv: Path, shap_matrix_2d: np.ndarray, feature_names: list[str], labels: list[str]) -> None:
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(shap_matrix_2d, index=feature_names, columns=labels).to_csv(out_csv)
    print("Saved SHAP matrix CSV ->", out_csv)

OUT_SHAP = FIG_DIR / "handover_event_shap_values.csv"
save_shap_matrix_csv(OUT_SHAP, shap_matrix, feature_cols, time_labels)


In [ ]:
# ============================================================
# 5) Save event heatmap PNG
# ============================================================
OUT_EVENT_PNG = FIG_DIR / "handover_event_heatmap.png"

plt.figure(figsize=(9, 4.8))
plt.imshow(shap_matrix, aspect="auto")
plt.yticks(np.arange(len(feature_cols)), feature_cols, fontsize=9)
plt.xticks(np.arange(len(time_labels)), time_labels, rotation=90, fontsize=7)
plt.title("Beam Handover – SHAP heatmap around highest-risk events (surrogate explanation)")
plt.tight_layout()
plt.savefig(OUT_EVENT_PNG, dpi=200, bbox_inches="tight")
plt.close()

print("Saved:", OUT_EVENT_PNG)


In [ ]:
# ============================================================
# 6) Continuous heatmap PNG (overview across events)
# ============================================================
OUT_CONT_PNG = FIG_DIR / "handover_continuous_heatmap.png"

# Subsample events for overview
rng = np.random.default_rng(42)
n_events = min(200, len(ho))
sel = np.sort(rng.choice(len(ho), size=n_events, replace=False))

X_sub = X_scaled[sel]
shap_sub = np.asarray(explainer.shap_values(X_sub))  # (events, features)

# Display all features (small set) across sampled events
cont = shap_sub.T
cont_time = [f"e{i}" for i in range(cont.shape[1])]

plt.figure(figsize=(9, 4.8))
plt.imshow(cont, aspect="auto")
plt.yticks(np.arange(len(feature_cols)), feature_cols, fontsize=9)
plt.xticks(np.arange(len(cont_time))[::20], cont_time[::20], rotation=90, fontsize=7)
plt.title("Beam Handover – Continuous SHAP overview (surrogate explanation)")
plt.tight_layout()
plt.savefig(OUT_CONT_PNG, dpi=200, bbox_inches="tight")
plt.close()

print("Saved:", OUT_CONT_PNG)


## Completion check
Verify these outputs exist:
- `data/processed/handover_table.csv`
- `reports/figures/handover_event_shap_values.csv`
- `reports/figures/handover_event_heatmap.png`
- `reports/figures/handover_continuous_heatmap.png`

After committing these outputs, the Streamlit dashboard should render the Beam Handover page without falling back to missing artefacts.